# UdaPlay 01 — RAG Pipeline

**Author: Sam Sepassi**

This notebook prepares a persistent ChromaDB vector store from the local
`games/` JSON dataset so the UdaPlay agent (notebook 02) can answer
questions via Retrieval-Augmented Generation.

Pipeline:

1. Load every `games/NNN.json` record
2. Convert each record into a single embeddable document
3. Embed and persist into ChromaDB (`chromadb/` directory)
4. Sanity-check with three semantic queries


## 1. Setup


In [1]:
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from lib.vector_store import VectorStoreManager, load_games_from_directory


## 2. Inspect the raw dataset


In [2]:
GAMES_DIR = Path('games')
files = sorted(GAMES_DIR.glob('*.json'))
print(f'Found {len(files)} game JSON files')
with files[0].open() as fp:
    print(json.dumps(json.load(fp), indent=2))


Found 20 game JSON files
{
  "Name": "Gran Turismo 3: A-Spec",
  "Platform": "PlayStation 2",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Developer": "Polyphony Digital",
  "YearOfRelease": 2001,
  "Description": "A realistic driving simulator featuring more than 150 licensed cars and a large roster of tracks. Praised at launch for its visuals and physics, it became one of the best-selling games on the PlayStation 2."
}


## 3. Convert JSON records into embeddable documents

`load_games_from_directory` reads each file and produces a `GameDocument`
whose `text` is a single natural-language summary — that's what the
embedder sees, so the document text deliberately includes name, year,
platform, developer, publisher, genre, and description.


In [3]:
documents = load_games_from_directory(GAMES_DIR)
print(f'Built {len(documents)} documents')
print('---')
print(documents[0].text)
print('---')
print(documents[0].metadata)


Built 20 documents
---
Gran Turismo 3: A-Spec (2001) — Racing on PlayStation 2. Developed by Polyphony Digital and published by Sony Computer Entertainment. A realistic driving simulator featuring more than 150 licensed cars and a large roster of tracks. Praised at launch for its visuals and physics, it became one of the best-selling games on the PlayStation 2.
---
{'name': 'Gran Turismo 3: A-Spec', 'platform': 'PlayStation 2', 'year': 2001, 'publisher': 'Sony Computer Entertainment', 'developer': 'Polyphony Digital', 'genre': 'Racing', 'source': 'games/001.json'}


## 4. Build the persistent vector store

ChromaDB writes everything to the `chromadb/` directory so the agent
notebook can re-use the same index without re-embedding. We `reset()`
first so re-runs are idempotent.


In [4]:
store = VectorStoreManager(persist_directory='chromadb')
store.reset()
written = store.add_games(documents)
print(f'Wrote {written} documents; collection now contains {store.count()} rows')


Wrote 20 documents; collection now contains 20 rows


Peek at the first few persisted docs to confirm the round-trip:


In [5]:
for row in list(store.peek(3)):
    print(row['id'], '->', row['metadata'].get('name'))


001 -> Gran Turismo 3: A-Spec
002 -> FIFA 21
003 -> God of War Ragnarok


## 5. Semantic search smoke tests

Three queries that map onto the project specification's example
questions. We print the top hit and its distance.


In [6]:
QUERIES = [
    'Who developed FIFA 21?',
    'When was God of War Ragnarok released?',
    'What platform was Pokemon Red launched on?',
]

for q in QUERIES:
    hits = store.query(q, k=3)
    print(f'Q: {q}')
    for h in hits:
        meta = h['metadata']
        print(f"  - {meta.get('name')} ({meta.get('year')}) [dist={h['distance']:.3f}]")
    print()


Q: Who developed FIFA 21?
  - FIFA 21 (2020) [dist=0.656]
  - Final Fantasy VII Remake (2020) [dist=1.485]
  - Pokemon Red (1996) [dist=1.518]

Q: When was God of War Ragnarok released?
  - God of War Ragnarok (2022) [dist=0.454]
  - Hades (2020) [dist=1.420]
  - Final Fantasy VII Remake (2020) [dist=1.523]



Q: What platform was Pokemon Red launched on?
  - Pokemon Red (1996) [dist=0.535]
  - Super Mario Odyssey (2017) [dist=1.244]
  - Red Dead Redemption 2 (2018) [dist=1.334]



## 6. Done

The vector store is now persisted to `chromadb/`. Move on to
`Udaplay_02_solution_project.ipynb` to run the agent.
